In [17]:
import os
import pandas as pd
from collections import defaultdict
from datetime import datetime
import json
import logger
import traceback

table_name_map = {
    "Ajia_plc_1.csv": "A架动作表.csv",
    "device_13_11_meter_1311.csv": "折臂吊车与小艇动作表.csv",
    "Port3_ksbg_9.csv": "艏推系统DP动作表.csv",
}

data_path = "assets/复赛数据/"
output_path = "data"

os.makedirs(output_path, exist_ok=True)

In [18]:
# 合并数据


# def merge_csv_files(input_path, out_path):
#     # 根据前缀分组文件
#     file_groups = defaultdict(list)
#     for file_name in os.listdir(input_path):
#         if file_name.endswith(".csv") and "字段释义" not in file_name:
#             prefix = file_name.rsplit("_", 1)[0]
#             file_groups[prefix].append(os.path.join(input_path, file_name))
#     # 合并前缀相同的文件
#     for prefix, file_list in file_groups.items():
#         merged_df = pd.concat(
#             (pd.read_csv(file) for file in file_list), ignore_index=True
#         )
#         output_file = os.path.join(out_path, f"{prefix}.csv")
#         os.makedirs(os.path.dirname(output_file), exist_ok=True)
#         merged_df.to_csv(output_file, index=False)
#         logger.info(f'合并前缀为"{prefix}"的文件到{output_file}')
#     # 将设备参数详情表转为csv
#     df_device = pd.read_excel(f"{data_path}设备参数详情.xlsx")
#     df_device.to_csv(os.path.join(out_path, "设备参数详情表.csv"), index=False)

# logger.special("开始合并数据")
# merge_csv_files(data_path, tmp_path)
# merge_csv_files(data_path, output_path)
# logger.success("数据合并完成")

In [19]:
# 判定A架的开关机和有无电流
table_key="Ajia_plc_1.csv"
logger.init()

def convert_to_numeric(value):
    """
    将值转换为数值类型，无法转换的返回 -1
    """
    try:
        return float(value)
    except ValueError:
        return -1


logger.special("开始判定A架开关机和有无电流")

df = pd.read_csv(os.path.join(data_path, table_key))
df = df.sort_values(by='csvTime')
df["Ajia-3_v"] = df["Ajia-3_v"].apply(convert_to_numeric)
df["Ajia-5_v"] = df["Ajia-5_v"].apply(convert_to_numeric)
df["status"] = "False"
df["check_current_presence"] = "False"
df["work_status"] = "未工作"
have_boot = -1
not_have_boot = -1

for i in range(1, df.shape[0]):
    prev_ajia3 = df.loc[i - 1, "Ajia-3_v"]
    prev_ajia5 = df.loc[i - 1, "Ajia-5_v"]
    curr_ajia3 = df.loc[i, "Ajia-3_v"]
    curr_ajia5 = df.loc[i, "Ajia-5_v"]

    # 停电条件：当前 Ajia-5_v == -1，且前一时刻 Ajia-5_v > 0 或 0
    if curr_ajia5 == -1 and (prev_ajia5 >= 0):
        df.loc[i, "status"] = "停电"

    # A架开机条件：前一时刻 Ajia-3_v == -1，且当前 Ajia-3_v >= 0
    if prev_ajia3 == -1 and curr_ajia3 >= 0:
        df.loc[i, "status"] = "A架开机"
        have_boot = i
    if prev_ajia5 == -1 and curr_ajia5 >= 0:
        df.loc[i, "status"] = "A架开机"
        have_boot = i

    # A架关机条件：当前 Ajia-3_v == -1，且前一时刻 Ajia-3_v >= 0
    if curr_ajia3 == -1 and prev_ajia3 >= 0:
        df.loc[i, "status"] = "A架关机"
        not_have_boot = i
    if curr_ajia5 == -1 and prev_ajia5 >= 0:
        df.loc[i, "status"] = "A架关机"
        not_have_boot = i

    if have_boot != -1 and not_have_boot != -1 and have_boot < not_have_boot:
        for j in range(have_boot, not_have_boot + 1):
            df.loc[j, "work_status"] = "开机工作中"
        have_boot = -1
        not_have_boot = -1

    # 有电流条件：前一时刻有一个或全部为0，下一刻均不为0
    if (prev_ajia3 <= 0 or prev_ajia5 <= 0) and (curr_ajia3 > 0 and curr_ajia5 > 0):
        df.loc[i, "check_current_presence"] = "有电流"
    # 无电流条件：前一时刻均不为0，下一刻有一个或全部为0
    elif prev_ajia3 > 0 and prev_ajia5 > 0 and (curr_ajia3 <= 0 or curr_ajia5 <= 0):
        df.loc[i, "check_current_presence"] = "无电流"

logger.success("A架开关机和有无电流判定完成")

2025-03-10 20:51:38.896 [SPECIAL] 开始判定A架开关机和有无电流
2025-03-10 20:51:45.093 [SUCCESS] A架开关机和有无电流判定完成


In [20]:
# 处理A架角度数据

def find_next_target_value(index, current_target):
    """
    从指定索引开始查找下一个完美摆动目标值
    """
    for i in range(index, df.shape[0]):
        if current_target == 35:
            value = df.loc[i, "Ajia-0_v"]
            if value == "error":
                continue
            value = float(value)
            if value > 30 and value < 38:
                df.loc[i, "full_swing"] = True
                print("完美摆动：",df.loc[i, "csvTime"],"Ajia-0_v:", df.loc[i, "Ajia-0_v"])
                return i, -43
        elif current_target == -43:
            value = df.loc[i, "Ajia-0_v"]
            if value == "error":
                continue
            value = float(value)
            if value < -40 and value > -46:
                df.loc[i, "full_swing"] = True
                print("完美摆动：",df.loc[i, "csvTime"],"Ajia-0_v:", df.loc[i, "Ajia-0_v"])
                return i, 35
    return -1, -1

def detect_swings(df):
    """
    假设A架右舷同一方向上摆动超过10°即可算作一次摆动,标记摆动

    """
    index=0
    for i in range(0, df.shape[0]):
        value = df.loc[i, "Ajia-0_v"]
        if value == "error":
            continue
        prev_value = float(df.loc[i, "Ajia-0_v"])
        index = i
        break
    
    curr_value = None
    for i in range(index+1, df.shape[0]):
        curr_value = df.loc[i, "Ajia-0_v"]
        if curr_value == "error":
            continue
        curr_value = float(curr_value)
        if prev_value*curr_value > 0 :
            if abs(curr_value-prev_value) > 10:
                df.loc[i, "directional_swing"] = True
                print("方向摆动超过10°：",df.loc[i, "csvTime"],"Ajia-0_v:", df.loc[i, "Ajia-0_v"])
                prev_value = curr_value
                continue
            elif abs(curr_value-prev_value) > 1.5 and prev_value< 30 and curr_value >30:
                prev_value = curr_value
                print("更新值为：",prev_value,"时间：",df.loc[i, "csvTime"])
                continue
        if prev_value*curr_value < 0:
            prev_value = curr_value
            continue
    print("方向摆动超过10°处理完成")
        

logger.special("开始处理A架角度范围")

current_target = None
first_target_index = None
first_target = None
df["full_swing"] = False
df["directional_swing"] = False
for i in range(0, df.shape[0]):
    value = df.loc[i, "Ajia-0_v"]
    if value == "error":
        continue
    value = float(value)
    if value < -40 and value > -46:
        first_target_index = i
        first_target = -43
        current_target = 35
        break
    elif value > 30 and value < 38:
        first_target_index = i
        first_target = 35
        current_target = -43
        break
index1 = first_target_index + 1
while index1 < df.shape[0]:
    value = df.loc[index1, "Ajia-0_v"]
    if value == "error":
        continue
    value = float(value)
    index1, current_target = find_next_target_value(index1, current_target)
    if index1 == -1:
        break
    index1 += 1
print("完美摆动处理完成")
detect_swings(df)
logger.success("A架角度范围处理完成")
# df = detect_swings(df)
# df.drop(columns=["Ajia-0_v_num"], inplace=True)
# print_surrounding_rows(df, "full_swing")
# print_surrounding_rows(df, "directional_swing")
# # df.to_csv(os.path.join(output_path, "test.csv"), index=False)
# logger.success("处理完成")

2025-03-10 20:51:45.125 [SPECIAL] 开始处理A架角度范围
完美摆动： 2024-05-17 10:00:50 Ajia-0_v: -43.2741
完美摆动： 2024-05-17 10:07:50 Ajia-0_v: 35.3355
完美摆动： 2024-05-17 19:00:50 Ajia-0_v: -42.0441
完美摆动： 2024-05-17 19:10:50 Ajia-0_v: 35.2684
完美摆动： 2024-05-18 08:24:50 Ajia-0_v: -43.0728
完美摆动： 2024-05-18 08:33:50 Ajia-0_v: 33.9489
完美摆动： 2024-05-18 16:22:50 Ajia-0_v: -42.3124
完美摆动： 2024-05-18 16:54:50 Ajia-0_v: 35.2013
完美摆动： 2024-05-19 08:25:50 Ajia-0_v: -43.4754
完美摆动： 2024-05-19 08:32:50 Ajia-0_v: 35.1118
完美摆动： 2024-05-19 16:21:50 Ajia-0_v: -43.1399
完美摆动： 2024-05-19 17:13:50 Ajia-0_v: 35.2684
完美摆动： 2024-05-20 07:17:50 Ajia-0_v: -43.5872
完美摆动： 2024-05-20 07:26:50 Ajia-0_v: 35.2684
完美摆动： 2024-05-22 16:23:49 Ajia-0_v: -41.6192
完美摆动： 2024-05-22 17:10:49 Ajia-0_v: 34.6869
完美摆动： 2024-05-22 23:00:49 Ajia-0_v: -42.0441
完美摆动： 2024-05-23 01:17:49 Ajia-0_v: 31.1087
完美摆动： 2024-05-23 11:25:49 Ajia-0_v: -43.8332
完美摆动： 2024-05-23 11:30:49 Ajia-0_v: 31.1087
完美摆动： 2024-05-23 17:00:49 Ajia-0_v: -45.6223
完美摆动： 2024-05-23 17:

In [21]:
# 检查Ajia-0_v摆动至最小值和最大值
def check_ajia_0_v_extremes(df):
    flag = False
    extremes = [0] * len(df)
    for i in range(0, len(df)):
        if df.loc[i, "Ajia-0_v"] == "error":
            extremes[i] = 0
            continue
        curr_ajia_0_v = float(df.loc[i, "Ajia-0_v"])
        if -44 <= curr_ajia_0_v <= -42 and flag == True:
            flag = False
            extremes[i] = -1
        elif 34 <= curr_ajia_0_v <= 36 and flag == False:
            flag = True
            extremes[i] = 1
        else :
            extremes[i] = 0
    return extremes

# df["ajia_0_v_extremes"] = check_ajia_0_v_extremes(df)

In [22]:
# 根据开关机事件，将A架数据分为若干段
logger.special("根据开关机事件，将A架数据分为若干段")
start_time = None
segments = []

for index, row in df.iterrows():
    if row["status"] == "A架开机":
        start_time = row["csvTime"]
    elif row["status"] == "A架关机" and start_time is not None:
        end_time = row["csvTime"]
        segments.append((start_time, end_time))
        start_time = None
        
logger.success("共分为%d段" % len(segments))
for i, (start_time, end_time) in enumerate(segments):
    logger.info(f"第{i+1}段：{start_time} - {end_time}")

2025-03-10 20:51:46.829 [SPECIAL] 根据开关机事件，将A架数据分为若干段
2025-03-10 20:51:49.952 [SUCCESS] 共分为204段
2025-03-10 20:51:49.952 [INFO] 第1段：2024-05-17 09:00:50 - 2024-05-17 10:15:50
2025-03-10 20:51:49.955 [INFO] 第2段：2024-05-17 19:00:50 - 2024-05-17 19:23:50
2025-03-10 20:51:49.955 [INFO] 第3段：2024-05-18 08:00:50 - 2024-05-18 09:12:50
2025-03-10 20:51:49.955 [INFO] 第4段：2024-05-18 13:31:50 - 2024-05-18 17:12:50
2025-03-10 20:51:49.996 [INFO] 第5段：2024-05-19 10:50:50 - 2024-05-19 15:39:50
2025-03-10 20:51:49.996 [INFO] 第6段：2024-05-19 15:41:50 - 2024-05-19 17:24:50
2025-03-10 20:51:49.996 [INFO] 第7段：2024-05-20 07:00:50 - 2024-05-20 07:35:50
2025-03-10 20:51:49.998 [INFO] 第8段：2024-05-20 14:00:50 - 2024-05-20 14:02:50
2025-03-10 20:51:49.998 [INFO] 第9段：2024-05-21 11:00:49 - 2024-05-21 12:23:49
2025-03-10 20:51:49.998 [INFO] 第10段：2024-05-21 17:12:49 - 2024-05-21 17:52:49
2025-03-10 20:51:49.998 [INFO] 第11段：2024-05-21 18:14:49 - 2024-05-21 18:20:49
2025-03-10 20:51:50.000 [INFO] 第12段：2024-05-22 08:00:49 

In [ ]:
# 让LLM预测不好判断的动作

prompt_ajia_judge_file = "prompts/ajia_judge.md"

XIAFANG = """你是一个细心的数据分析助手，请根据给定的电流变化序列数据，准确返回三个值。  

规则要求：  
1. 识别最后一段非零数据，该段应至少包含两次升降（即电流从约 56 上升至 70 以上）。  
2. 结果应从最后一段非零数据中选择，且满足以下条件：  
   - 第一个值：该段的第一个峰值，且一般大于 70。  
   - 第二个值：位于第一个和第三个值之间，一般小于 60。  
   - 第三个值：重新达到峰值，且一般大于 70。  
3. 忽略大于 200 的异常数据。  
4. 数据可能存在噪声，请谨慎判断。思考完成后，不需要返回思考过程，以列表形式返回三个值,回答中只有列表。
5. 若无法找到符合条件的三个值，请返回 `[-100, -100, -100]`，不要随意捏造。  

示例：  
输入：  
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.6478, 56.5133, 60.8637, 56.3751, 56.3777, 56.3601, 61.1564, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 422.499, 56.2896, 66.3951, 60.8928, 57.7813, 56.3871, 66.3077, 62.5263, 56.3937, 58.0826, 90.0969, 87.5592, 83.9934, 56.5033, 59.3441, 58.0018, 56.3027, 56.2845, 56.3666, 101.763, 96.6118, 56.3492, 59.2629, 57.0112, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]  
输出：  
[90.0969, 56.5033, 101.763]  

现有一组新的电流变化序列数据： 
<<L>>  

请根据上述规则，返回符合要求的三个值，答案仅包含列表格式。"""


HUISHOU = """你是一个细心的数据分析助手，请根据给定的电流变化序列数据，准确返回三个值。  

规则要求：  
1. 数据列表应包含至少两段非零数据。  
2. 结果应满足以下条件：  
   - 第一个值：来自非最后一段非零数据的峰值，且一般大于 70。  
   - 第二个值：来自最后一段非零数据的峰值，且一般应大于 70。  
   - 第三个值：位于第二个值之后，且一般小于 60，即最后一个峰值回落至低于 60 的点。  
3. 忽略大于 200 的异常数据。  
4. 数据可能存在噪声，请谨慎判断。思考完成后，不需要返回思考过程，以列表形式返回三个值,回答中只有列表。  
5. 若无法找到符合条件的三个值，请返回 `[-100, -100, -100]`，不要随意捏造。  

示例：  
输入：  
[0.0, 0.0, 0.0, 0.0, 0.0, 57.0048, 56.8545, 61.9802, 56.8646, 56.8705, 56.777, 68.3751, 56.5526, 56.6556, 63.1736, 68.4542, 78.2151, 86.3214, 82.7017, 58.9111, 56.632, 56.9142, 56.6583, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.2542, 56.2177, 56.1263, 56.2697, 56.102, 59.5568, 57.5703, 57.6415, 56.9307, 57.0531, 56.9337, 58.582, 58.0159, 104.238, 96.6301, 97.1496, 56.5543, 63.426, 57.6552, 56.6086, 56.6611, 56.5601, 56.6476, 56.68, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]  
输出：  
[86.3214, 104.238, 56.5543]  

现有一组新的电流变化序列数据：  
<<L>>  

请根据上述规则，返回符合要求的三个值，答案仅包含列表格式。"""


def predict_sequence_by_llm(L_sequence, is_xiafang: bool):
    from api import get_completion
    from utils import parse_res,load_api_config
    load_api_config('GLM')

    # with open(prompt_ajia_judge_file, "r", encoding="utf-8") as file:
    #     ajia_judge = file.read()

    if is_xiafang:
        ajia_judge = XIAFANG
    else:
        ajia_judge = HUISHOU

    prompt = ajia_judge.replace("<<L>>", L_sequence)
    messages = [{"role": "user", "content": prompt}]
    response = get_completion(messages)
    res = parse_res(response)
    logger.info("【LLM返回】：%s" % res)
    return res


def single_predict(L_sequence, is_xiafang: bool):
    result_list = json.loads(predict_sequence_by_llm(L_sequence, is_xiafang))
    if len(result_list) == 3:
        a = result_list[0]
        b = result_list[1]
        c = result_list[2]
        return a, b, c


def get_predict_result(L_sequence, is_xiafang: bool):
    try:
        return single_predict(L_sequence, is_xiafang)
    except Exception as e:
        logger.error("LLM预测失败，尝试再次预测：%s" % e)
        try:
            return single_predict(L_sequence, is_xiafang)
        except Exception as e:
            logger.error("LLM预测失败，返回默认值：%s" % e)
            return -100, -100, -100

In [24]:
# 判断A架关键动作的辅助函数
def extract_daily_power_on_times(df):
    """
    从CSV文件中提取一天内有两次开机的第一次和第二次开机时间。

    参数:
    file_path (str): CSV文件的路径，包含 'csvTime' 和 'status' 列。

    返回:
    first_start_times (list): 一天内有两次开机的第一次开机时间列表。
    second_start_times (list): 一天内有两次开机的第二次开机时间列表。
    """
    df["csvTime"] = pd.to_datetime(df["csvTime"])

    df["date"] = df["csvTime"].dt.date

    daily_segments = {}

    for date, group in df.groupby("date"):
        segments = []
        start_time = None

        for index, row in group.iterrows():
            if row["status"] == "A架开机":
                start_time = row["csvTime"]
            elif row["status"] == "A架关机" and start_time is not None:
                end_time = row["csvTime"]
                segments.append((start_time, end_time))
                start_time = None

        daily_segments[date] = segments

    daily_counts = {date: len(segments) for date, segments in daily_segments.items()}

    two_times_days = [date for date, count in daily_counts.items() if count == 2]

    first_start_times = []
    second_start_times = []

    for date in two_times_days:
        segments = daily_segments[date]
        first_start_times.append(segments[0][0])
        second_start_times.append(segments[1][0])

    return first_start_times, second_start_times


def find_peaks(input_data):
    """
    找到峰值

    :param input_data:输入序列
    :return: 峰值数量，峰值列表
    """
    data = [50 if 50 <= num <= 66 else num for num in input_data]

    # 找到峰值
    peaks = []
    for i in range(1, len(data) - 1):  # 从第二个元素遍历到倒数第二个元素
        if data[i] > data[i - 1] and data[i] > data[i + 1]:  # 判断是否为峰值
            peaks.append(data[i])  # 只记录峰值值
    peaks = [peak for peak in peaks if peak > 75]
    # 返回峰值格式和具体的峰值
    return len(peaks), peaks


def find_first_increasing_value(data):
    """
    找到列表中第一个从稳定值（66以下）开始增加的值

    :param data: 输入的数值列表
    :return: 第一个大于66的值。如果未找到，返回50
    """
    processed_data = [50 if 50 <= num <= 66 else num for num in data]

    for _, value in enumerate(processed_data):
        if value > 66 and value < 300:
            return value
    return 50


def find_stable_value(data1, data2, peak1, peak2):
    """
    找到两个峰值之间的数据中，回落到稳定值的第一个值。
    假设稳定值在 50 到 60 之间。

    :param data1 (list): 数据列表
    :param data2 (list): 数据列表
    :param peak1 (float): 第一个峰值
    :param peak2 (float): 第二个峰值

    :return float or None: 稳定值，如果未找到则返回 None
    """
    try:
        start_index = data1.index(peak1)
        end_index = data1.index(peak2)
    except ValueError:
        return None

    between_peaks1 = data1[start_index : end_index + 1]
    between_peaks2 = data2[start_index : end_index + 1]

    for index, value in enumerate(between_peaks1):
        if 50 <= value <= 60 and 50 <= between_peaks2[index] <= 60:
            return value

    return None


def find_first_stable_after_peak(data, peak, stable_min=50, stable_max=60):
    """
    从峰值到列表末尾的数据中，找到第一个回落到稳定值的值。

    :param data (list): 数据列表
    :param peak (float): 峰值
    :param stable_min (float): 稳定值的最小值
    :param stable_max (float): 稳定值的最大值

    :return float or None: 稳定值，如果未找到则返回 None
    """
    try:
        start_index = data.index(peak)
    except ValueError:
        return None

    after_peak = data[start_index:]

    for value in after_peak:
        if stable_min <= value <= stable_max:
            return value

    return None


def extract_peak_pattern(current_presence_data):
    """
    从数据中提取峰值模式\n
    事件对：从有电流到无电流是一个

    :param current_presence_data: 有无电流数据
    :return: 返回时间段内各个事件对内的峰值数量
    """
    logger.info(f"【提取事件对】事件数量: {current_presence_data.shape[0]}")
    peak_pattern = []
    if current_presence_data.shape[0] >= 2 and current_presence_data.shape[0] % 2 == 0:
        for i in range(0, current_presence_data.shape[0], 2):
            event_start = current_presence_data.iloc[i]
            event_end = current_presence_data.iloc[i + 1]
            # 确保第一个事件是“有电流”，第二个事件是“无电流”
            if (
                event_start["check_current_presence"] == "有电流"
                and event_end["check_current_presence"] == "无电流"
            ):
                event_start_time = event_start["csvTime"]
                event_end_time = event_end["csvTime"]
                between_data = df[
                    (df["csvTime"] >= event_start_time)
                    & (df["csvTime"] <= event_end_time)
                ]
                ajia_5_data = list(between_data["Ajia-5_v"])
                logger.info(
                    f"【提取事件对】事件对 ({i}, {i + 1}) 之间的数据: {ajia_5_data}"
                )
                len_peaks, peak_L = find_peaks(ajia_5_data)
                logger.info(
                    f"【提取事件对】事件对 ({i}, {i + 1}) 之间的峰值数量: {len_peaks}，峰值为{peak_L}"
                )
                peak_pattern.append(len_peaks)
    return peak_pattern

In [26]:
# 判定A架的关键动作
# 提取每个区段内的“通电流”和“关电流”事件


class PredictResult:
    def __init__(self, start_time, end_time):
        """
        预测结果类
        :param start_time: 起始时间
        :param end_time: 结束时间
        :param prediction: 预测结果
        """
        self.start_time = start_time
        self.end_time = end_time
        self.prediction: list[float] = None

    def __str__(self):
        return f"预测时间段: {self.start_time} - {self.end_time}, 预测结果: {self.prediction}"


LLM_predict_count = 0
LLM_predict_results: dict[int, PredictResult] = {}
for segment in segments:
    start, end = segment
    logger.success(f"【开始处理时间段】开机时间: {start}, 关机时间: {end}")
    logger.info(f"【处理时间段】开始提取事件对")
    segment_data = df[(df["csvTime"] >= start) & (df["csvTime"] <= end)]
    logger.info(f"【处理时间段】区间数据：{list(segment_data['Ajia-5_v'])}")
    current_presence_data = df[
        (df["csvTime"] >= start)
        & (df["csvTime"] <= end)
        & (df["check_current_presence"].isin(["有电流", "无电流"]))
    ]
    peak_pattern = extract_peak_pattern(current_presence_data)
    logger.info(f"【处理时间段】区间类型：{peak_pattern}")
    if (
        peak_pattern == [2]
        or peak_pattern == [0, 2]
        or peak_pattern == [0, 0, 2]
        or peak_pattern == [0, 3]
        or peak_pattern == [0, 1, 3]
        or peak_pattern == [1, 3]
    ):
        # 下放阶段
        logger.info(f"【处理时间段】下放阶段")
        if peak_pattern == [2]:
            event_start_time = current_presence_data.iloc[0]["csvTime"]
            event_end_time = current_presence_data.iloc[1]["csvTime"]
        elif peak_pattern == [0, 2] or peak_pattern == [0, 3] or peak_pattern == [1, 3]:
            event_start_time = current_presence_data.iloc[2]["csvTime"]
            event_end_time = current_presence_data.iloc[3]["csvTime"]
        elif peak_pattern == [0, 0, 2] or peak_pattern == [0, 1, 3]:
            event_start_time = current_presence_data.iloc[4]["csvTime"]
            event_end_time = current_presence_data.iloc[5]["csvTime"]
        between_data = df[
            (df["csvTime"] >= event_start_time) & (df["csvTime"] <= event_end_time)
        ]
        ajia_5_data = list(between_data["Ajia-5_v"])
        ajia_3_data = list(between_data["Ajia-3_v"])
        len_peaks, peak_L = find_peaks(ajia_5_data)
        # 征服者起吊：电流从稳定值（50多），取高于50的点
        first_increasing_value = find_first_increasing_value(ajia_5_data)
        indices = between_data.index[
            between_data["Ajia-5_v"] == first_increasing_value
        ].tolist()
        df.loc[indices, "status"] = "征服者起吊"
        # 缆绳解除：电流从高值回落至稳定值（50多），取50
        stable_value = find_stable_value(
            ajia_5_data, ajia_3_data, peak_L[len_peaks - 2], peak_L[len_peaks - 1]
        )
        indices = between_data.index[between_data["Ajia-5_v"] == stable_value].tolist()
        df.loc[indices, "status"] = "缆绳解除"
        # 征服者入水：缆绳解除的时间点往前推一分钟
        previous_indices = [idx - 1 for idx in indices if idx > 0]
        df.loc[previous_indices, "status"] = "征服者入水"
        # A架摆回：征服者入水后，电流重新增加到峰值（最大值点）
        indices = between_data.index[
            between_data["Ajia-5_v"] == peak_L[len_peaks - 1]
        ].tolist()
        df.loc[indices, "status"] = "A架摆回"
        df.loc[df["csvTime"] == start, "work_status"] = "布放阶段开始"
        df.loc[df["csvTime"] == end, "work_status"] = "布放阶段结束"
        df.loc[(df["csvTime"] > start) & (df["csvTime"] < end), "work_status"] = "布放阶段中"
    elif peak_pattern == [1, 2] or peak_pattern == [1, 1]:
        # 回收阶段
        logger.info(f"【处理时间段】回收阶段")
        # 第一个事件对
        event_start_time = current_presence_data.iloc[0]["csvTime"]
        event_end_time = current_presence_data.iloc[1]["csvTime"]
        between_data = df[
            (df["csvTime"] >= event_start_time) & (df["csvTime"] <= event_end_time)
        ]
        ajia_5_data = list(between_data["Ajia-5_v"])

        len_peaks, peak_L = find_peaks(ajia_5_data)
        # A架摆出：征服者起吊前，电流到达峰值（取峰值）
        first_increasing_value = find_first_increasing_value(ajia_5_data)
        indices = between_data.index[between_data["Ajia-5_v"] == peak_L[0]].tolist()
        df.loc[indices, "status"] = "A架摆出"
        # 第二个事件对
        event_start_time = current_presence_data.iloc[2]["csvTime"]
        event_end_time = current_presence_data.iloc[3]["csvTime"]
        between_data = df[
            (df["csvTime"] >= event_start_time) & (df["csvTime"] <= event_end_time)
        ]
        ajia_5_data = list(between_data["Ajia-5_v"])

        len_peaks, peak_L = find_peaks(ajia_5_data)
        max_value = max([x for x in ajia_5_data if x <= 200])

        # 征服者出水：电流峰值（取峰值）
        indices = between_data.index[between_data["Ajia-5_v"] == max_value].tolist()
        df.loc[indices, "status"] = "征服者出水"
        # 缆绳挂妥：征服者出水往前推一分钟
        previous_indices = [idx - 1 for idx in indices if idx > 0]
        df.loc[previous_indices, "status"] = "缆绳挂妥"
        # 征服者落座：电流从高值回落至稳定值（50多）（取50）
        first_stable_after_peak = find_first_stable_after_peak(ajia_5_data, max_value)
        indices = between_data.index[
            between_data["Ajia-5_v"] == first_stable_after_peak
        ].tolist()
        df.loc[indices, "status"] = "征服者落座"
        df.loc[df["csvTime"] == start, "work_status"] = "回收阶段开始"
        df.loc[df["csvTime"] == end, "work_status"] = "回收阶段结束"
        df.loc[(df["csvTime"] > start) & (df["csvTime"] < end), "work_status"] = "回收阶段中"
    elif len(peak_pattern) > 0:
        LLM_predict_count += 1
        LLM_predict_results[LLM_predict_count] = PredictResult(start, end)
        logger.info("【处理时间段】交由大模型预测")

        segment_data = segment_data.copy()
        segment_data.loc[:, "csvTime"] = pd.to_datetime(segment_data["csvTime"])
        # 获取第一个值
        first_value = segment_data["csvTime"].iloc[0]
        # 判断小时是否大于12点
        is_hour_greater_than_12 = first_value.hour > 12
        first_start_times, second_start_times = extract_daily_power_on_times(df=df)

        def predict(data, is_xiafang):
            segment_data["predict_column"] = data.apply(
                lambda row: (
                    row["Ajia-3_v"]
                    if row["Ajia-5_v"] == 0 and row["Ajia-3_v"] > 0
                    else row["Ajia-5_v"]
                ),
                axis=1,
            )
            logger.info(
                "【处理时间段】----------------LLM预测的列表---------------------"
            )
            logger.info(str(list(segment_data["predict_column"])))
            try:
                a, b, c = get_predict_result(
                    str(list(segment_data["predict_column"])), is_xiafang
                )
            except Exception as e:
                logger.error(f"An error occurred: {e}\n{traceback.format_exc()}")
                a, b, c = -100, -100, -100
            LLM_predict_results[LLM_predict_count].prediction = (a, b, c)
            logger.success(
                "【处理时间段】LLM预测结果：",
                "下放阶段" if is_xiafang else "回收阶段",
                a,
                b,
                c,
            )
            if is_xiafang:
                indices = segment_data.index[
                    segment_data["predict_column"] == a
                ].tolist()
                df.loc[indices, "status"] = "征服者起吊"

                indices = segment_data.index[
                    segment_data["predict_column"] == b
                ].tolist()
                df.loc[indices, "status"] = "缆绳解除"
                previous_indices = [idx - 1 for idx in indices if idx > 0]
                df.loc[previous_indices, "status"] = "征服者入水"

                indices = segment_data.index[
                    segment_data["predict_column"] == c
                ].tolist()
                df.loc[indices, "status"] = "A架摆回"
                df.loc[df["csvTime"] == start, "work_status"] = "布放阶段开始"
                df.loc[df["csvTime"] == end, "work_status"] = "布放阶段结束"
                df.loc[(df["csvTime"] > start) & (df["csvTime"] < end), "work_status"] = "布放阶段中"
            else:
                indices = segment_data.index[
                    segment_data["predict_column"] == a
                ].tolist()
                df.loc[indices, "status"] = "A架摆出"

                indices = segment_data.index[
                    segment_data["predict_column"] == b
                ].tolist()
                df.loc[indices, "status"] = "征服者出水"
                previous_indices = [idx - 1 for idx in indices if idx > 0]
                df.loc[previous_indices, "status"] = "缆绳挂妥"

                indices = segment_data.index[
                    segment_data["predict_column"] == c
                ].tolist()
                df.loc[indices, "status"] = "征服者落座"
                df.loc[df["csvTime"] == start, "work_status"] = "回收阶段开始"
                df.loc[df["csvTime"] == end, "work_status"] = "回收阶段结束"
                df.loc[(df["csvTime"] > start) & (df["csvTime"] < end), "work_status"] = "回收阶段中"

        if is_hour_greater_than_12:
            predict(segment_data, False)
        elif first_value in first_start_times:
            predict(segment_data, True)
        elif first_value in second_start_times:
            predict(segment_data, False)

    logger.success(f"【处理时间段完成】开机时间: {start}, 关机时间: {end}")

2025-03-10 20:56:03.781 [SUCCESS] 【开始处理时间段】开机时间: 2024-05-17 09:00:50, 关机时间: 2024-05-17 10:15:50
2025-03-10 20:56:03.781 [INFO] 【处理时间段】开始提取事件对
2025-03-10 20:56:03.784 [INFO] 【处理时间段】区间数据：[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 55.9764, 56.7053, 56.5347, 56.0214, 91.1025, 56.4408, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.1899, 56.1375, 56.0303, 56.2232, 56.0663, 56.1043, 55.4824, 88.3943, 88.2448, 85.7528, 80.5977, 99.5829, 56.1955, 55.9012, 56.1437, 65.2289, 102.213, 98.5886, 110.858, 77.0694, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]
2025-03-10 20:56:03.787 [INFO] 【提取事件对】事件数量: 5
2025-03-10 20:56:03.787 [INFO] 【处理时间段】区间类型：[]
2025-03-10 20:56:03.788 [SUCCESS] 【处理时间段完成】开机时间: 2024-05-17 09:00:50, 关机时间: 2024-05-17 10:15:50
2025-03-10 20:56:03.788 [SUCCESS] 【开始处理时间段】开机时间: 2024-05-17 19:00:50, 关机时间: 2024-05-17 19:23:50
2025-03-10 20:56:03.789 [INFO] 【处理时间段】开始提取事件对
2025

KeyboardInterrupt: 

In [ ]:
print("LLM预测总数：", LLM_predict_count)
for key, value in LLM_predict_results.items():
    print(f"{key}: {value}")
with open(f"{output_path}/LLM_predict_time_range.txt", "w") as f:
    for key, value in LLM_predict_results.items():
        f.write(f"{key}: {value}\n")

LLM预测总数： 27
1: 预测时间段: 2024-05-17 19:00:50 - 2024-05-17 19:23:50, 预测结果: (86.9921, 106.473, 56.4856)
2: 预测时间段: 2024-05-18 09:33:50 - 2024-05-18 17:12:50, 预测结果: (89.5143, 109.118, 56.4596)
3: 预测时间段: 2024-05-19 07:30:50 - 2024-05-19 15:39:50, 预测结果: (94.0032, 56.5145, 108.417)
4: 预测时间段: 2024-05-22 15:53:49 - 2024-05-23 01:46:49, 预测结果: (89.5498, 121.124, 56.4044)
5: 预测时间段: 2024-05-23 07:56:49 - 2024-05-23 11:35:49, 预测结果: (91.9004, 55.9536, 104.662)
6: 预测时间段: 2024-05-23 17:00:49 - 2024-05-23 22:46:49, 预测结果: (-100, -100, -100)
7: 预测时间段: 2024-05-23 23:05:49 - 2024-05-24 14:00:49, 预测结果: (88.7477, 91.2588, 56.4319)
8: 预测时间段: 2024-05-24 16:00:49 - 2024-05-24 23:32:49, 预测结果: (121.933, 121.667, 56.5639)
9: 预测时间段: 2024-05-25 07:56:49 - 2024-05-25 09:10:49, 预测结果: None
10: 预测时间段: 2024-05-26 14:54:48 - 2024-05-26 17:23:48, 预测结果: (94.1761, 102.783, 56.2041)
11: 预测时间段: 2024-05-27 07:47:48 - 2024-05-27 09:47:48, 预测结果: (89.1991, 57.012, 104.007)
12: 预测时间段: 2024-05-27 17:00:48 - 2024-05-27 19:21:48, 预测结果: (9

In [ ]:
# 保存A架数据
logger.special("开始保存A架数据")
df = df.drop(columns=["date"])
# df = df.drop(columns=['check_current_presence'])
df.to_csv(os.path.join(output_path, table_key), index=False)
df.to_csv(os.path.join(output_path, table_name_map[table_key]), index=False)
logger.success("A架数据保存成功")

2025-02-28 15:24:31.530 [SPECIAL] 开始保存A架数据
2025-02-28 15:24:31.890 [SUCCESS] A架数据保存成功


In [ ]:
# 判定ON DP和OFF DP
table_key = "Port3_ksbg_9.csv"
logger.special("开始判定ON DP和OFF DP")
df = pd.read_csv(os.path.join(output_path, table_key))
df["P3_33"] = pd.to_numeric(df["P3_33"], errors="coerce")
df["status"] = "False"
df["work_status"]= "未开机"
have_boot = -1
not_have_boot = -1

for i in range(1, df.shape[0]):
    # ON DP
    if df.loc[i - 1, "P3_33"] == 0 and df.loc[i, "P3_33"] > 0:
        df.loc[i, "status"] = "ON DP"
        have_boot = i
    # OFF DP
    if df.loc[i - 1, "P3_33"] > 0 and df.loc[i, "P3_33"] == 0:
        df.loc[i, "status"] = "OFF DP"
        not_have_boot = i
    if have_boot != -1 and not_have_boot != -1 and have_boot < not_have_boot:
        for j in range(have_boot, not_have_boot):
            df.loc[j, "work_status"] = "开机工作中"
        have_boot = -1
        not_have_boot = -1
        
df.to_csv(os.path.join(output_path, table_key), index=False)
df.to_csv(os.path.join(output_path, table_name_map[table_key]), index=False)
logger.success("ON DP和OFF DP数据保存成功")

2025-02-28 15:24:31.911 [SPECIAL] 开始判定ON DP和OFF DP
2025-02-28 15:24:33.287 [SUCCESS] ON DP和OFF DP数据保存成功


In [ ]:
# 处理折臂吊车
from collections import Counter
table_key = "device_13_11_meter_1311.csv"

logger.special("开始判定折臂吊车关键动作")
df = pd.read_csv(os.path.join(data_path, table_key))
df["13-11-6_v"] = pd.to_numeric(df["13-11-6_v"], errors="coerce")
df["status"] = "False"
df["action"] = "False"


def sliding_window_5(arr):
    """滑动窗口大小为5的逻辑"""
    window_size = 5
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i : i + window_size]
        if (
            window[1] < 10
            and window[2] < 10
            and window[3] < 10
            and window[0] > 10
            and window[4] > 10
        ):
            # 将 window[0] 包装成列表进行赋值
            modified_arr[i + 1 : i + 4] = [window[0]] * 3
    return modified_arr


def sliding_window_4(arr):
    """滑动窗口大小为4的逻辑"""
    window_size = 4
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i : i + window_size]
        if window[1] < 10 and window[2] < 10 and window[0] > 10 and window[3] > 10:
            # 将 window[0] 包装成列表进行赋值
            modified_arr[i + 1 : i + 3] = [window[0]] * 2
    return modified_arr


def sliding_window_3(arr):
    """滑动窗口大小为3的逻辑"""
    window_size = 3
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i : i + window_size]
        if window[1] < 10 and window[0] > 10 and window[2] > 10:
            # 直接赋值，因为只修改一个值
            modified_arr[i + 1] = window[0]
    return modified_arr


logger.info("【处理折臂吊车】开始应用滑动窗口逻辑")
df["13-11-6_v_new"] = sliding_window_5(df["13-11-6_v"].tolist())
df["13-11-6_v_new"] = sliding_window_4(df["13-11-6_v_new"].tolist())
df["13-11-6_v_new"] = sliding_window_3(df["13-11-6_v_new"].tolist())
logger.success("【处理折臂吊车】滑动窗口逻辑应用完成")

logger.info("【处理折臂吊车】开始判定折臂吊车的开机和关机事件")
df["work_status"]= "未工作"
have_boot = -1
not_have_boot = -1
for i in range(1, df.shape[0]):
    # 开机
    if df.iloc[i - 1]["13-11-6_v"] == 0 and df.iloc[i]["13-11-6_v"] > 0:
        df.at[df.index[i], "status"] = "折臂吊车开机"
        have_boot = i
    # 关机
    if df.iloc[i - 1]["13-11-6_v"] > 0 and df.iloc[i]["13-11-6_v"] == 0:
        df.at[df.index[i], "status"] = "折臂吊车关机"
        not_have_boot = i
    if have_boot != -1 and not_have_boot != -1 and have_boot < not_have_boot:
        for j in range(have_boot, not_have_boot+1):
            df.loc[j, "work_status"] = "开机工作中"
        have_boot = -1
        not_have_boot = -1
    # 检测由待机进入工作和由工作进入待机的事件
    if df.iloc[i - 1]["13-11-6_v_new"] < 10 and df.iloc[i]["13-11-6_v_new"] > 10:
        df.at[df.index[i], "action"] = "由待机进入工作"
    if df.iloc[i - 1]["13-11-6_v_new"] > 10 and df.iloc[i]["13-11-6_v_new"] < 10:
        df.at[df.index[i], "action"] = "由工作进入待机"
logger.success("【处理折臂吊车】折臂吊车的开机和关机事件判定完成")

logger.info("【处理折臂吊车】根据折臂吊车的开机和关机事件划分时间段")
segments = []
start_time = None
for index, row in df.iterrows():
    if row["status"] == "折臂吊车开机":
        start_time = row["csvTime"]
    elif row["status"] == "折臂吊车关机" and start_time is not None:
        end_time = row["csvTime"]
        segments.append((start_time, end_time))
        start_time = None
logger.success("【处理折臂吊车】时间段划分完成")

def find_most_frequent_number(lst):
    """
    使用 Counter 统计每个数的出现次数\n
    找到出现次数最多的数（如果有多个，只返回第一个）
    """
    counter = Counter(lst)
    most_common_number = counter.most_common(1)[0][0]
    return most_common_number

for segment in segments:
    start, end = segment
    logger.info(f"【开始处理时间段】开始时间：{start}，结束时间：{end}")
    actions_data = df[
        (df["csvTime"] >= start)
        & (df["csvTime"] <= end)
        & (df["action"].isin(["由待机进入工作", "由工作进入待机"]))
    ]
    segment_data = df[(df["csvTime"] >= start) & (df["csvTime"] <= end)]
    # 检查事件数量是否为偶数且等于6
    if actions_data.shape[0] > 0 and actions_data.iloc[0]["csvTime"] == start:
        actions_data = actions_data[2:]
        segment_data = segment_data[2:]
    if actions_data.shape[0] == 8:
        csv_time_as_datetime = pd.to_datetime(actions_data["csvTime"], errors="coerce")
        # 计算时间差
        time_diffs = (csv_time_as_datetime.iloc[1::2].values - csv_time_as_datetime.iloc[::2].values).astype("timedelta64[s]").astype(int)
        # 找到最小时间差的位置
        min_idx = time_diffs.argmin() * 2
        # 直接 drop 对应索引
        actions_data = actions_data.drop(actions_data.index[[min_idx, min_idx + 1]])
        
    logger.info(f"【处理时间段】事件数量: {actions_data.shape[0]}")
    if actions_data.shape[0] == 6:
        # 处理每一对事件
        for i in range(0, 6, 2):
            event_start = actions_data.iloc[i]
            event_end = actions_data.iloc[i + 1]

            if (
                event_start["action"] == "由待机进入工作"
                and event_end["action"] == "由工作进入待机"
            ):
                event_start_time = event_start["csvTime"]
                event_end_time = event_end["csvTime"]
                between_data = df[
                    (df["csvTime"] >= event_start_time)
                    & (df["csvTime"] <= event_end_time)
                ]
                ajia_5_data = list(between_data["13-11-6_v"])

                # 找到最后一个大于9的值
                last_value_above_9 = next((x for x in reversed(ajia_5_data) if x > 9), None)

                if last_value_above_9 is not None:
                    all_indices = between_data.index[
                        between_data["13-11-6_v_new"] == last_value_above_9
                    ].tolist()
                    last_index = all_indices[-1] if all_indices else None

                    # 根据事件对的顺序更新status
                    if last_index is not None:
                        if i == 0:
                            df.loc[last_index, "status"] = "小艇检查完毕"
                        elif i == 2:
                            df.loc[last_index, "status"] = "小艇入水"
                        elif i == 4:
                            df.loc[last_index, "status"] = "小艇落座"
                else:
                    logger.info("列表中没有大于 9 的值")
    if actions_data.shape[0] == 4:
        # 处理每一对事件
        for i in range(0, 4, 2):
            event_start = actions_data.iloc[i]
            event_end = actions_data.iloc[i + 1]
            if (
                event_start["action"] == "由待机进入工作"
                and event_end["action"] == "由工作进入待机"
            ):
                event_start_time = event_start["csvTime"]
                event_end_time = event_end["csvTime"]
                between_data = df[
                    (df["csvTime"] >= event_start_time)
                    & (df["csvTime"] <= event_end_time)
                ]
                ajia_5_data = list(between_data["13-11-6_v"])

                # 找到最后一个大于9的值
                last_value_above_9 = next((x for x in reversed(ajia_5_data) if x > 9), None)

                if last_value_above_9 is not None:
                    all_indices = between_data.index[
                        between_data["13-11-6_v_new"] == last_value_above_9
                    ].tolist()
                    last_index = all_indices[-1] if all_indices else None

                    # 根据事件对的顺序更新status
                    if (
                        last_index is not None
                        and df.loc[last_index, "status"] == "False"
                    ):
                        if i == 0:
                            df.loc[last_index, "status"] = "小艇入水"
                        elif i == 2:
                            df.loc[last_index, "status"] = "小艇落座"
                else:
                    print("列表中没有大于 9 的值")
                # 保存结果
# df = df.drop(columns=['action'])
df = df.drop(columns=['13-11-6_v_new'])
df.to_csv(os.path.join(output_path, table_key), index=False)
df.to_csv(os.path.join(output_path, table_name_map[table_key]), index=False)
logger.success("【处理折臂吊车】保存数据完成")

2025-02-28 15:24:33.310 [SPECIAL] 开始判定折臂吊车关键动作
2025-02-28 15:24:33.353 [INFO] 【处理折臂吊车】开始应用滑动窗口逻辑
2025-02-28 15:24:33.388 [SUCCESS] 【处理折臂吊车】滑动窗口逻辑应用完成
2025-02-28 15:24:33.388 [INFO] 【处理折臂吊车】开始判定折臂吊车的开机和关机事件
2025-02-28 15:24:41.648 [SUCCESS] 【处理折臂吊车】折臂吊车的开机和关机事件判定完成
2025-02-28 15:24:41.648 [INFO] 【处理折臂吊车】根据折臂吊车的开机和关机事件划分时间段
2025-02-28 15:24:42.655 [SUCCESS] 【处理折臂吊车】时间段划分完成
2025-02-28 15:24:42.655 [INFO] 【开始处理时间段】开始时间：2024-05-16 22:14:41，结束时间：2024-05-16 23:40:06
2025-02-28 15:24:42.661 [INFO] 【处理时间段】事件数量: 6
2025-02-28 15:24:42.673 [INFO] 【开始处理时间段】开始时间：2024-05-17 08:55:43，结束时间：2024-05-17 10:10:18
2025-02-28 15:24:42.679 [INFO] 【处理时间段】事件数量: 2
2025-02-28 15:24:42.679 [INFO] 【开始处理时间段】开始时间：2024-05-17 19:00:27，结束时间：2024-05-17 19:18:32
2025-02-28 15:24:42.686 [INFO] 【处理时间段】事件数量: 2
2025-02-28 15:24:42.686 [INFO] 【开始处理时间段】开始时间：2024-05-18 08:00:34，结束时间：2024-05-18 08:50:01
2025-02-28 15:24:42.692 [INFO] 【处理时间段】事件数量: 6
2025-02-28 15:24:42.702 [INFO] 【开始处理时间段】开始时间：2024-05-18 16:07:42，结束时间：2024-05-18 1